# Task 2 - Incremental CPG Parser Service

The service uses Python `ast`, stable structural IDs,
statement-level CFG, lexical reaching-definitions DFG, and conservative
top-level same-file call resolution. It releases each file graph before
processing the next file.

```mermaid
flowchart LR
  F[One Python file] --> A[Python ast]
  A --> N[AST nodes and edges]
  A --> C[Statement CFG]
  C --> D[Reaching-definitions DFG]
  A --> K[Conservative CALL edges]
  N --> T[One Kafka transaction]
  C --> T
  D --> T
  K --> T
  T --> M[Manifest advances after commit]
```

## Approach and rationale

**Approach:** Each file is decoded, analyzed, reconciled against its previous
stable node and edge IDs, and emitted in one Kafka transaction. AST field/index
paths define identity; the CFG models statement flow; DFG uses a bounded
per-scope fixed point; unresolved definitions and calls become explicit external
nodes rather than guessed targets.

**Why this approach:** The standard-library AST is available without a heavy
Joern runtime and preserves every Python syntax node needed by the lab.
File-local passes bound graph memory by the largest source file. Structural IDs,
stale deletes, and updating SQLite only after Kafka commits jointly make retries
idempotent and recoverable.

**Alternatives and trade-offs:** Joern offers deeper interprocedural semantics
and tree-sitter offers robust multi-version parsing, but both add integration
cost beyond the laboratory scope. The chosen analysis deliberately sacrifices
alias analysis, precise exception flow, and dynamic dispatch; warnings and
external nodes expose those limits instead of overstating accuracy.

In [1]:
import json
import sys
from collections import Counter
from pathlib import Path

root = Path('..').resolve()
sys.path.insert(0, str(root))
from cpg_parser.analyzer import CPGAnalyzer
from cpg_parser.discovery import discover_repo
from cpg_parser.ids import file_id

repo = root / 'source-repo'
if not repo.is_dir():
    repo = root.parent / 'source-repo'
report = discover_repo(repo)
node_counts, edge_counts = Counter(), Counter()
node_ids, edge_ids = set(), set()
warnings = 0
for relative in report.files:
    result = CPGAnalyzer(
        (repo / relative).read_text(encoding='utf-8', errors='replace'),
        file_id('huggingface/optimum', relative), relative,
    ).analyze()
    node_counts.update(result.node_counts())
    edge_counts.update(result.edge_counts())
    node_ids.update(node.id for node in result.nodes)
    edge_ids.update(edge.id for edge in result.edges)
    warnings += len(result.warnings)
summary = {
    'repository': 'huggingface/optimum',
    'files': len(report.files),
    'nodes': sum(node_counts.values()),
    'edges': sum(edge_counts.values()),
    'node_counts': dict(sorted(node_counts.items())),
    'edge_counts': dict(sorted(edge_counts.items())),
    'warnings': warnings,
    'all_node_ids_unique': len(node_ids) == sum(node_counts.values()),
    'all_edge_ids_unique': len(edge_ids) == sum(edge_counts.values()),
}
print(json.dumps(summary, indent=2))
assert {'AST', 'CFG', 'DFG', 'CALL'} <= set(edge_counts)
assert summary['all_node_ids_unique'] and summary['all_edge_ids_unique']
print('PASS: all CPG categories exist and IDs are unique')

{
  "repository": "huggingface/optimum",
  "files": 61,
  "nodes": 62550,
  "edges": 77873,
  "node_counts": {
    "AST": 58021,
    "EXTERNAL": 3047,
    "SYNTHETIC": 1482
  },
  "edge_counts": {
    "AST": 57960,
    "CALL": 2593,
    "CFG": 7248,
    "DFG": 10072
  },
  "warnings": 31,
  "all_node_ids_unique": true,
  "all_edge_ids_unique": true
}
PASS: all CPG categories exist and IDs are unique


In [2]:
import subprocess
import sys
from pathlib import Path

result = subprocess.run(
    [sys.executable, '-m', 'pytest', '-q'],
    cwd=Path('..').resolve(), capture_output=True, text=True, check=True,
)
print(result.stdout.rstrip())
assert '[100%]' in result.stdout
print('PASS: parser, replay, schema, and syntax-error tests')

.........................                                                [100%]
PASS: parser, replay, schema, and syntax-error tests


## Reflection

**Worked:** Deterministic IDs, all four CPG edge categories, bounded file-by-file processing, and stale-element reconciliation pass the regression suite.

**Issue encountered during development:** Syntax errors originally risked leaving the last valid graph in Neo4j, while attribute calls and `AugAssign`/`del` produced misleading static-analysis results.

**Resolution:** Error transactions now delete stale elements and update the manifest; dynamic attribute calls remain external, and DFG transfer functions explicitly model augmented reads and deletion kills. Aliasing and runtime dispatch remain documented limits.